In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/trips_clean.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1447937 entries, 0 to 1447936
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1447937 non-null  str    
 1   vendor_id           1447937 non-null  int64  
 2   pickup_datetime     1447937 non-null  str    
 3   dropoff_datetime    1447937 non-null  str    
 4   passenger_count     1447937 non-null  int64  
 5   pickup_longitude    1447937 non-null  float64
 6   pickup_latitude     1447937 non-null  float64
 7   dropoff_longitude   1447937 non-null  float64
 8   dropoff_latitude    1447937 non-null  float64
 9   store_and_fwd_flag  1447937 non-null  str    
 10  trip_duration       1447937 non-null  int64  
dtypes: float64(4), int64(3), str(4)
memory usage: 187.8 MB


In [2]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"]) 
df['dropoff_datetime'] = pd.to_datetime(df["dropoff_datetime"])


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1447937 entries, 0 to 1447936
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1447937 non-null  str           
 1   vendor_id           1447937 non-null  int64         
 2   pickup_datetime     1447937 non-null  datetime64[us]
 3   dropoff_datetime    1447937 non-null  datetime64[us]
 4   passenger_count     1447937 non-null  int64         
 5   pickup_longitude    1447937 non-null  float64       
 6   pickup_latitude     1447937 non-null  float64       
 7   dropoff_longitude   1447937 non-null  float64       
 8   dropoff_latitude    1447937 non-null  float64       
 9   store_and_fwd_flag  1447937 non-null  str           
 10  trip_duration       1447937 non-null  int64         
dtypes: datetime64[us](2), float64(4), int64(3), str(2)
memory usage: 135.3 MB


In [4]:
df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [1]:
# Kiểm tra số chuyến dùng để TRAIN và hai nhóm giờ trong tập TEST
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
processed = root / "data" / "processed"
trips = pd.read_csv(processed / "trips_clean.csv",
                    usecols=["id", "pickup_datetime", "dropoff_datetime"],
                    parse_dates=["pickup_datetime", "dropoff_datetime"])

# 1. Chia theo giờ xuất phát. Khi bắt đầu kiểm thử, chỉ học từ chuyến đã kết thúc.
trips = trips.sort_values("pickup_datetime")
cut = int(len(trips) * 0.8)
test = trips.iloc[cut:].set_index("id")
cutoff = test["pickup_datetime"].min()
before = trips.iloc[:cut]
train = before.loc[before["dropoff_datetime"] < cutoff]

# 2. Đếm theo giờ xuất phát của từng chuyến test (giờ New York).
pickup = test["pickup_datetime"].dt
peak = (pickup.dayofweek < 5) & pickup.hour.isin([6, 7, 8, 9, 16, 17, 18, 19])
counts = {"overall": len(test), "peak": int(peak.sum()), "off_peak": int((~peak).sum())}

# 3. Đối chiếu mã chuyến, cờ cao điểm và số mẫu với các file do train.py xuất.
saved_train = pd.read_csv(processed / "train.csv", usecols=["id"])
saved_test = pd.read_csv(processed / "test.csv", usecols=["id", "is_rush_hour"]).set_index("id")
assert sorted(train["id"]) == sorted(saved_train["id"]), "Danh sách chuyến train khác nhau"
assert peak.astype(int).sort_index().equals(saved_test["is_rush_hour"].sort_index()), "Tập test hoặc cờ cao điểm khác nhau"
metrics = pd.read_csv(root / "reports" / "metrics.csv")
for group, n in counts.items():
    assert set(metrics.loc[metrics["group"] == group, "n"]) == {n}, f"Số mẫu khác metrics.csv: {group}"

print(f"Dữ liệu sau lọc: {len(trips):,} chuyến | Mốc bắt đầu test: {cutoff}")
print(f"Train: {cut:,} chuyến ban đầu − {len(before) - len(train)} chuyến chưa có nhãn tại mốc chia = {len(train):,}")
print(f"Test: {counts['overall']:,} = {counts['peak']:,} cao điểm + {counts['off_peak']:,} ngoài cao điểm")
print("Đã đối chiếu khớp với train.csv, test.csv và metrics.csv.")

Dữ liệu sau lọc: 1,447,937 chuyến | Mốc bắt đầu test: 2016-05-24 01:07:16
Train: 1,158,349 chuyến ban đầu − 27 chuyến chưa có nhãn tại mốc chia = 1,158,322
Test: 289,588 = 85,210 cao điểm + 204,378 ngoài cao điểm
Đã đối chiếu khớp với train.csv, test.csv và metrics.csv.
